# 02 Extração e Filtragem do Léxico

Léxico extraído APENAS do treino. Teste permanece isolado.

In [3]:
import json
from pathlib import Path
from collections import Counter
import sys
sys.path.insert(0, str(Path('..').resolve()))

from src.data_loading import load_ddsmall
from src.lexicon import extract_lexicon, filter_lexicon

ROOT = Path('..')
train = load_ddsmall(ROOT / 'data/raw/DDsmall/train/dd_corpus_small_train.json')

raw_index = extract_lexicon(train)
lexicon   = filter_lexicon(raw_index)

print(f'Entidades brutas (únicas): {len(raw_index)}')
print(f'Entidades após filtragem:  {len(lexicon)}')
print(f'Removidas: {len(raw_index) - len(lexicon)}')

Entidades brutas (únicas): 5899
Entidades após filtragem:  1057
Removidas: 4842


In [4]:
# Análise distribuição por classe
by_class = Counter(v['label'] for v in lexicon.values())
print('Por classe:', dict(by_class))

# Entidades mais frequentes por classe
for label in ['Person', 'Location', 'Organization']:
    top = sorted(
        [(k, v) for k, v in lexicon.items() if v['label'] == label],
        key=lambda x: x[1]['freq'], reverse=True
    )[:5]
    print(f'\nTop {label}:')
    for key, info in top:
        print(f"  '{info['text']}' — freq={info['freq']} docs={info['doc_count']}")

Por classe: {'Organization': 147, 'Location': 662, 'Person': 248}

Top Person:
  'Lucio Claudio Lima' — freq=38 docs=20
  'Ruan' — freq=34 docs=27
  'Rafael' — freq=33 docs=25
  'Rodrigo' — freq=28 docs=24
  'Mara' — freq=26 docs=24

Top Location:
  'rj' — freq=217 docs=184
  'São Gonçalo' — freq=192 docs=177
  'favelinha' — freq=155 docs=128
  'Tijuca' — freq=137 docs=39
  'Rio de Janeiro' — freq=101 docs=92

Top Organization:
  'polícia' — freq=600 docs=486
  'Facebook' — freq=100 docs=76
  'pm' — freq=86 docs=78
  'Batalhão' — freq=70 docs=64
  'bpm' — freq=63 docs=60


In [5]:
# Exemplos de entidades removidas
removed = {k: v for k, v in raw_index.items() if k not in lexicon}
removed_sorted = sorted(removed.items(), key=lambda x: x[1]['freq'], reverse=True)
print(f'Entidades removidas ({len(removed_sorted)}), ordenadas por frequência:')
for k, v in removed_sorted:
    print(f"  '{k}' — classes={dict(v['class_counts'])} freq={v['freq']}")

Entidades removidas (4842), ordenadas por frequência:
  'flamengo' — classes={'Organization': 6, 'Location': 16} freq=22
  'botafogo' — classes={'Location': 9, 'Organization': 7, 'Person': 2} freq=18
  'sg' — classes={'Location': 15} freq=15
  'p2' — classes={'Organization': 14, 'Person': 1} freq=15
  'ph' — classes={'Person': 12} freq=12
  'jacare' — classes={'Person': 5, 'Location': 6} freq=11
  'dl' — classes={'Person': 11} freq=11
  'sp' — classes={'Location': 10} freq=10
  'abraxas' — classes={'Organization': 7, 'Person': 1, 'Location': 1} freq=9
  'oi' — classes={'Organization': 8} freq=8
  'mg' — classes={'Location': 8} freq=8
  'dh' — classes={'Organization': 8} freq=8
  'vk' — classes={'Person': 1, 'Location': 6} freq=7
  'ipase' — classes={'Organization': 2, 'Location': 4} freq=6
  'brizolao' — classes={'Organization': 4, 'Location': 2} freq=6
  'jordao' — classes={'Location': 3, 'Person': 3} freq=6
  'cangulo' — classes={'Location': 3, 'Person': 1} freq=4
  'maxwell' — class

In [6]:
# Salvar léxico filtrado
out = ROOT / 'data/lexicon/lexicon_filtered.json'
with open(out, 'w', encoding='utf-8') as f:
    json.dump(lexicon, f, ensure_ascii=False, indent=2)
print(f'Léxico salvo em {out}')

Léxico salvo em ..\data\lexicon\lexicon_filtered.json
